In [1]:
import numpy as np
import pandas as pd
import ast
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
movies = pd.read_csv('/kaggle/input/tmdb-movie-metadata/tmdb_5000_movies.csv')
credits = pd.read_csv('/kaggle/input/tmdb-movie-metadata/tmdb_5000_credits.csv') 

In [3]:
movies = movies.merge(credits,on='title')

In [4]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]
movies.dropna(inplace=True)

In [5]:
def convert(text):
    try:
        return [i['name'] for i in ast.literal_eval(text)]
    except:
        return []

In [6]:
movies.dropna(inplace=True)

In [7]:
def convert3(text):
    try:
        return [i['name'] for i in ast.literal_eval(text)[:3]]
    except:
        return []

In [8]:
def fetch_director(text):
    try:
        return [i['name'] for i in ast.literal_eval(text) if i['job'] == 'Director']
    except:
        return []

In [9]:
def collapse(L):
    return [i.replace(" ", "") for i in L]

In [10]:
def convert3(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
        counter+=1
    return L 

In [11]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert3)
movies['crew'] = movies['crew'].apply(fetch_director)

In [12]:
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)


In [13]:
movies['overview'] = movies['overview'].apply(lambda x: x.split() if isinstance(x, str) else [])
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
movies = movies[['movie_id', 'title', 'tags']]

In [14]:
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))


In [15]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
vector = tfidf.fit_transform(movies['tags']).toarray()

In [16]:
similarity = cosine_similarity(vector)

In [23]:
pickle.dump(movies, open('/kaggle/working/movie_list.pkl', 'wb'))
pickle.dump(similarity, open('/kaggle/working/similarity.pkl', 'wb'))

In [18]:
movies = pickle.load(open('/kaggle/working/movie_list.pkl', 'rb'))
similarity = pickle.load(open('/kaggle/working/similarity.pkl', 'rb'))

In [19]:
def recommend(movie):
    if movie not in movies['title'].values:
        return ["Movie not found"]
    
    index = movies[movies['title'] == movie].index[0]
    distances = sorted(enumerate(similarity[index]), reverse=True, key=lambda x: x[1])
    
    recommended_movies = [movies.iloc[i[0]].title for i in distances[1:6]]
    
    return recommended_movies

In [22]:
print(recommend("Batman"))


['Batman', 'Batman & Robin', 'Batman Returns', 'The Dark Knight Rises', 'Batman Begins']
